In [1]:
from transformers import AutoTokenizer , AutoModelForSequenceClassification
import torch
import re

In [2]:
tokenizer = AutoTokenizer.from_pretrained('ealvaradob/bert-finetuned-phishing')

model = AutoModelForSequenceClassification.from_pretrained('ealvaradob/bert-finetuned-phishing' , num_labels=3 , ignore_mismatched_sizes=True)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ealvaradob/bert-finetuned-phishing and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([2, 1024]) in the checkpoint and torch.Size([3, 1024]) in the model instantiated
- classifier.bias: found shape torch.Size([2]) in the checkpoint and torch.Size([3]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [3]:
import pandas as pd
df1 = pd.read_csv("spam.csv", encoding="latin-1")

In [4]:
df1.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [5]:
df1 = df1.loc[:, ~df1.columns.str.contains('^Unnamed')] 

In [6]:
df1.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
df1 = df1.rename(columns = {"v1" : "label" , "v2" : "text"})

In [8]:
from datasets import load_dataset

dataset = load_dataset("ealvaradob/phishing-dataset", "combined_reduced")

In [9]:
df4= dataset['train'].to_pandas()

In [10]:
df4

,text,label
0,<!doctypehtml><html lang=en xml:lang=en xmlns=...,0
1,http://online0mgeving.ga/triodos/,1
2,metronews.ca/webapp/Login.aspx?logout=true&rur...,0
3,https://rarkuntem.co.jp.fpjiehk.cn/index1.php,1
4,freebase.com/view/m/0ct05qn,0
...,...,...
77672,http://www.definitions.net/definition/icon,0
77673,schedule crawler : hourahead failure start dat...,0
77674,"<!doctypehtml PUBLIC ""-//W3C//DTD HTML+RDFa 1....",0
77675,amazon.com/Sound-Horror-James-Philbrook/dp/B00...,0


In [11]:
df4["label"] = df4["label"].replace(1 , "phising")
df4["label"] = df4["label"].replace(0 , "benign")

In [12]:
df1["label"] = df1["label"].replace("ham" , "benign")

In [13]:
df = pd.concat([df1 , df4] , ignore_index=True)

In [14]:
df["label"] = df["label"].replace("benign" , 0)
df["label"] = df["label"].replace("phising" , 1)
df["label"] = df["label"].replace("spam" , 2)

C:\Users\Paras Rana\AppData\Local\Temp\ipykernel_2796\1015246566.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["label"] = df["label"].replace("spam" , 2)


In [15]:
df

,label,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,2,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
83244,0,http://www.definitions.net/definition/icon
83245,0,schedule crawler : hourahead failure start dat...
83246,0,"<!doctypehtml PUBLIC ""-//W3C//DTD HTML+RDFa 1...."
83247,0,amazon.com/Sound-Horror-James-Philbrook/dp/B00...


In [16]:
from sklearn.model_selection import train_test_split

train_df , test_df = train_test_split(df , test_size=0.2, stratify=df["label"], random_state=42)

In [17]:
from datasets import Dataset
train_ds = Dataset.from_pandas(train_df)
test_ds = Dataset.from_pandas(test_df)

In [18]:
train_df

,label,text
51733,0,facebook.com/people/Gary-Moeller/100000395720071
65592,0,ca.linkedin.com/in/iducharme
26964,0,Matthias Saou (matthias@egwn.net) wrote*:\n>Th...
44911,1,www769.paypal.co.uk.13340.ssl-340.mx/js/webapp...
38996,0,<!doctypehtml><html lang=en-CA><meta charset=u...
...,...,...
32900,1,booltom.com/7dp0k
2814,0,Some are lasting as much as 2 hours. You might...
53995,0,daylife.com/topic/Ella_Baker
59589,0,https://sneakernews.com/tag/under-armour-curry-3/


In [19]:
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

In [20]:
train_ds = train_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)

Map:   0%|          | 0/66599 [00:00<?, ? examples/s]

Map:   0%|          | 0/16650 [00:00<?, ? examples/s]

In [21]:
train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

In [22]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
)

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {"f1": (preds == labels).mean()}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

C:\Users\Paras Rana\AppData\Local\Temp\ipykernel_2796\2391644983.py:9: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
C:\Anaconda\envs\sih_env\lib\site-packages\torch\utils\data\dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss


In [ ]:
def predict_sentence(sentence, model, tokenizer):
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    
    model.eval()
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
    
    pred_class = torch.argmax(logits, dim=1).item()
    
    return pred_class

In [ ]:
sentence = "This is a test sentence."
predicted_class = predict_sentence(sentence, model, tokenizer)